In [2]:
import pandas as pd
import sqlite3

In [3]:
df= pd.read_csv("HR-Employee-Attrition.csv")
conn = sqlite3.connect("HR-Employee-attrition.db")
df.to_sql("employee_attrition", conn, if_exists="replace", index=False)
print("Database created successfully.")

Database created successfully.


In [6]:
query = """
SELECT *
FROM employee_attrition
LIMIT 5;
"""

pd.read_sql(query, conn)

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


# QUESTION 1: How many employees are there in the company?

In [7]:
query = """
SELECT COUNT(*) AS total_employees
FROM employee_attrition;
"""
pd.read_sql(query, conn)

,total_employees
0,1470


# QUESTION 2: How many employees left the company?

In [9]:
query = """
SELECT Attrition, COUNT(*) AS Employee_Count
FROM employee_attrition
GROUP BY Attrition;
"""
pd.read_sql(query, conn)

,Attrition,Employee_Count
0,No,1233
1,Yes,237


# QUESTION 3: Which departements have the highest number of employees?

In [10]:
query = """
SELECT Department, COUNT(*) AS Employee_Count
FROM employee_attrition
GROUP BY Department
ORDER BY Employee_Count DESC;
"""
pd.read_sql(query, conn)

,Department,Employee_Count
0,Research & Development,961
1,Sales,446
2,Human Resources,63


# Insight
research and development has the largest workforce.Prioritize HR strategies in large departments to maximize impact

# QUESTION 4: Which job roles have the highest attrition?

In [11]:
query = """
SELECT JobRole, COUNT(*) AS Employee_Left
FROM employee_attrition
WHERE Attrition= 'Yes'
GROUP BY JobRole
ORDER BY Employee_Left DESC;
"""
pd.read_sql(query, conn)

,JobRole,Employee_Left
0,Laboratory Technician,62
1,Sales Executive,57
2,Research Scientist,47
3,Sales Representative,33
4,Human Resources,12
5,Manufacturing Director,10
6,Healthcare Representative,9
7,Manager,5
8,Research Director,2


# Insight 
sales representative and labrotary technician gave the highest attrition. Focus retention efforts on high-turnover job roles

# QUESTION 5: Which departemants have the highest attrition?

In [12]:
query = """
SELECT Department, COUNT(*) AS Employee_Left
FROM employee_attrition
WHERE Attrition= 'Yes'
GROUP BY Department
ORDER BY Employee_Left DESC;
"""
pd.read_sql(query, conn)

,Department,Employee_Left
0,Research & Development,133
1,Sales,92
2,Human Resources,12


# Insight
Research & Development has te highest number of employess leaving. 

# QUESTION 6: Does overtime affect employee attrition?

In [13]:
query = """
SELECT OverTime, COUNT(*) AS Employee_Left
FROM employee_attrition
WHERE Attrition= 'Yes'
GROUP BY OverTime
ORDER BY Employee_Left DESC;
"""
pd.read_sql(query, conn)

,OverTime,Employee_Left
0,Yes,127
1,No,110


# Insight
Employee working overtime are more likely to leave. Reduce excessive overtime to improve employee retention

# QUESTION 7: What is the average monthly income by job role?

In [16]:
query = """
SELECT JobRole, AVG(MonthlyIncome) AS AVG_Monthly_Income
FROM employee_attrition
GROUP BY JobRole
ORDER BY AVG_Monthly_Income DESC;
"""
pd.read_sql(query, conn)

,JobRole,AVG_Monthly_Income
0,Manager,17181.676471
1,Research Director,16033.550000
2,Healthcare Representative,7528.763359
3,Manufacturing Director,7295.137931
4,Sales Executive,6924.279141
5,Human Resources,4235.750000
6,Research Scientist,3239.972603
7,Laboratory Technician,3237.169884
8,Sales Representative,2626.000000


# Insight:
Managers have the highest average monthly income.

# QUESTION 8: What is the average job satisfaction by departement?

In [17]:
query = """
SELECT Department, AVG(JobSatisfaction) AS AVG_Job_Satisfaction
FROM employee_attrition
GROUP BY Department
ORDER BY AVG_Job_Satisfaction DESC;
"""
pd.read_sql(query, conn)

,Department,AVG_Job_Satisfaction
0,Sales,2.751121
1,Research & Development,2.726327
2,Human Resources,2.603175


# Insight
Job satisfaction is simillar across departments, with slight differences.

# QUESTION 9: What is the average work-life balance by department?

In [18]:
query = """
SELECT Department, AVG(WorkLifeBalance) AS AVG_WorkLife_Balance
FROM employee_attrition
GROUP BY Department
ORDER BY AVG_Worklife_Balance DESC;
"""
pd.read_sql(query, conn)

,Department,AVG_WorkLife_Balance
0,Human Resources,2.920635
1,Sales,2.816143
2,Research & Development,2.725286


# Insight: 
Work-life balance is fairly consiste across departments

# QUESTION 10: Which business travel category has the highest attrition?

In [19]:
query = """
SELECT BusinessTravel, COUNT(*) AS Employees_Left
FROM employee_attrition
WHERE Attrition= 'Yes'
GROUP BY BusinessTravel
ORDER BY Employees_Left DESC;
"""
pd.read_sql(query, conn)

,BusinessTravel,Employees_Left
0,Travel_Rarely,156
1,Travel_Frequently,69
2,Non-Travel,12


# Insight
Employees who travel frequently have higher attrition

# QUESTION 11: Which job roles have an average monthly income above 7,000 $?

In [20]:
query = """
SELECT JobRole, AVG(MonthlyIncome) AS AVG_Monthly_Income
FROM employee_attrition
GROUP BY JobRole
HAVING AVG (MonthlyIncome) > 7000
ORDER BY AVG_Monthly_Income DESC;
"""
pd.read_sql(query, conn)

,JobRole,AVG_Monthly_Income
0,Manager,17181.676471
1,Research Director,16033.550000
2,Healthcare Representative,7528.763359
3,Manufacturing Director,7295.137931


# Insight
Only a few job roles have an average monthly income above $7000

# QUESTION 12: Which employees earn more than the company average salary?

In [21]:
query = """
WITH AvgSalary AS (SELECT AVG(MonthlyIncome) AS Company_Avg FROM employee_attrition)
SELECT EmployeeNumber, JobRole, MonthlyIncome
FROM employee_attrition, AvgSalary
WHERE MonthlyIncome > Company_Avg
ORDER BY MonthlyIncome DESC;
"""
pd.read_sql(query, conn)

,EmployeeNumber,JobRole,MonthlyIncome
0,259,Manager,19999
1,1035,Research Director,19973
2,1191,Manager,19943
3,226,Manager,19926
4,787,Manager,19859
...,...,...,...
488,496,Healthcare Representative,6540
489,523,Sales Executive,6538
490,889,Sales Executive,6524
491,1724,Manufacturing Director,6516


# Insight:
493 employees earn above the company average salary

# QUESTION 13: How do employees rank by monthly income within each job role?

In [23]:
query = """
SELECT EmployeeNumber,JobRole,MonthlyIncome, RANK() OVER (PARTITION BY JobRole
ORDER BY MonthlyIncome DESC) AS Income_Rank
FROM employee_attrition;
"""
pd.read_sql(query, conn)

,EmployeeNumber,JobRole,MonthlyIncome,Income_Rank
0,1661,Healthcare Representative,13966,1
1,431,Healthcare Representative,13964,2
2,258,Healthcare Representative,13734,3
3,1278,Healthcare Representative,13577,4
4,119,Healthcare Representative,13503,5
...,...,...,...,...
1465,411,Sales Representative,1200,79
1466,1273,Sales Representative,1118,80
1467,1928,Sales Representative,1091,81
1468,1876,Sales Representative,1081,82


# Insight 
Employees are ranked by income whithin each job role

# QUESTION 14: Which age groups have the highest attrition?

In [24]:
query = """
SELECT CASE
     WHEN Age < 30 THEN 'Under 30'
     WHEN Age BETWEEN 30 AND 39 THEN '30-39'
     WHEN Age BETWEEN 40 AND 49 THEN '40-49'
     ELSE '50+'
   END AS Age_Group,
   COUNT(*) AS Employees_Left
FROM employee_attrition
WHERE Attrition = 'Yes'
GROUP BY Age_Group
ORDER BY Employees_Left DESC;
"""
pd.read_sql(query, conn)

,Age_Group,Employees_Left
0,Under 30,91
1,30-39,89
2,40-49,34
3,50+,23


# Insight:
Employees under 30 have the highest attrition.